Magesha R.P -
Minor project 1

GroupDNA your WhatsApp group chat, decoded.

Feature 1: Chat Parser

In [1]:
import numpy as np
from datetime import datetime, timedelta

# Define the Google Colab file path
file_path = '/content/hostel_bois.txt'

# Initialization counters
system_messages_count = 0
media_omitted_count = 0
deleted_messages_count = 0

parsed_messages = []
media_per_person = {}
deleted_per_person = {}

def is_date_pattern(s):
    # Checks if line begins with a valid date format DD/MM/YY
    if len(s) < 8:
        return False
    return (s[0:2].isdigit() and s[2] == '/' and
            s[3:5].isdigit() and s[5] == '/' and
            s[6:8].isdigit())

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

current_message = None

for line in lines:
    line_str = line.strip('\r\n')
    if not line_str.strip():
        continue # Skip empty lines

    # Multi-line handling: if it doesn't start with a date pattern, append to previous message
    if not is_date_pattern(line_str):
        if current_message is not None:
            current_message['text'] += " " + line_str
        continue

    try:
        # Separate date-time from the rest of the text content
        timestamp_part, rest = line_str.split(' - ', 1) if ' - ' in line_str else line_str.split(' ', 1)

        # Format commas cleanly out of timestamps if present
        if ',' in timestamp_part:
            timestamp_part = timestamp_part.replace(',', '').strip()

        # Detect System Messages (lines without a colon splitting sender and text)
        if ':' not in rest:
            system_messages_count += 1
            continue

        sender_part, text_part = rest.split(':', 1)
        sender = sender_part.strip()
        text = text_part.strip()

        # Handle Deleted Messages
        if text == "This message was deleted":
            deleted_messages_count += 1
            deleted_per_person[sender] = deleted_per_person.get(sender, 0) + 1
            continue

        # Handle Media Omitted
        if text == "<Media omitted>":
            media_omitted_count += 1
            media_per_person[sender] = media_per_person.get(sender, 0) + 1
            # Still append to main interactions so activity volume balances out

        if current_message is not None:
            parsed_messages.append(current_message)

        current_message = {
            'timestamp': timestamp_part.strip(),
            'sender': sender,
            'text': text
        }
    except Exception:
        system_messages_count += 1

if current_message is not None:
    parsed_messages.append(current_message)

print(f"Successfully parsed {len(parsed_messages)} messages from 6 participants.")
print(f"Skipped: {system_messages_count} system messages, {media_omitted_count} media-omitted, {deleted_messages_count} deleted messages.")

Successfully parsed 3159 messages from 6 participants.
Skipped: 4 system messages, 32 media-omitted, 15 deleted messages.


Feature 2 – Group Overview

In [2]:
msg_counts = {}
for msg in parsed_messages:
    sender = msg['sender']
    msg_counts[sender] = msg_counts.get(sender, 0) + 1

# Sorted breakdown tracking by descending frequency
sorted_participants = sorted(msg_counts.items(), key=lambda x: x[1], reverse=True)
total_messages = len(parsed_messages)
unique_members = [p[0] for p in sorted_participants]

first_date_str = parsed_messages[0]['timestamp'].split(' ')[0]
last_date_str = parsed_messages[-1]['timestamp'].split(' ')[0]

d1 = datetime.strptime(parsed_messages[0]['timestamp'], '%d/%m/%y %H:%M')
d2 = datetime.strptime(parsed_messages[-1]['timestamp'], '%d/%m/%y %H:%M')
total_days = (d2.date() - d1.date()).days + 1

print(f"Period: {first_date_str} to {last_date_str} ({total_days} days)")
print(f"Total processed messages: {total_messages}")
for name, num in sorted_participants:
    print(f"- {name}: {num} messages ({ (num/total_messages)*100 :.1f}%)")

Period: 01/04/24 to 30/05/24 (60 days)
Total processed messages: 3159
- Rahul: 947 messages (30.0%)
- Priya: 716 messages (22.7%)
- Neha: 632 messages (20.0%)
- Aman: 488 messages (15.4%)
- Karan: 352 messages (11.1%)
- Vikas: 24 messages (0.8%)


Feature 3 — Most Active Day and Hour

In [3]:
day_activity = {}
hour_activity = {}

for msg in parsed_messages:
    date_part, time_part = msg['timestamp'].split(' ')
    hour_part = time_part.split(':')[0]

    day_activity[date_part] = day_activity.get(date_part, 0) + 1
    hour_activity[hour_part] = hour_activity.get(hour_part, 0) + 1

busiest_day = max(day_activity.items(), key=lambda x: x[1])
busiest_hour = max(hour_activity.items(), key=lambda x: x[1])

print(f"Busiest Day: {busiest_day[0]} ({busiest_day[1]} messages)")
print(f"Busiest Hour: {busiest_hour[0]}:00 ({busiest_hour[1]} messages across dataset)")

Busiest Day: 04/05/24 (76 messages)
Busiest Hour: 18:00 (246 messages across dataset)


Feature 4 — Activity Heatmap (NumPy Matrix)

In [4]:
member_to_idx = {name: i for i, name in enumerate(unique_members)}
heatmap_matrix = np.zeros((len(unique_members), 24))

for msg in parsed_messages:
    sender = msg['sender']
    _, time_part = msg['timestamp'].split(' ')
    hour_val = int(time_part.split(':')[0])

    if sender in member_to_idx:
        heatmap_matrix[member_to_idx[sender], hour_val] += 1

print("Successfully initialized and populated a 6x24 NumPy activity heatmap.")

Successfully initialized and populated a 6x24 NumPy activity heatmap.


Feature 5 — Top Words

In [11]:
stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'this', 'that', 'it', 'you', 'was'}
punctuation_chars = '.,!?()[]{}"\'_ -;:'

word_counts = {}
for msg in parsed_messages:
    if msg['text'] == "<Media omitted>":
        continue
    words = msg['text'].lower().split()
    for w in words:
        cleaned_w = w.strip(punctuation_chars)
        if cleaned_w and cleaned_w not in stop_words:
            word_counts[cleaned_w] = word_counts.get(cleaned_w, 0) + 1

sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 Group Words Captured:")
for w, c in sorted_words:
    print(f"- {w}: {c} times")

Top 5 Group Words Captured:
- how: 321 times
- guys: 318 times
- so: 292 times
- about: 274 times
- hai: 268 times


Feature 6 — Response Speed & Silent Streaks

In [5]:
response_gaps = {name: [] for name in unique_members}
last_sender = None
last_time = None

for msg in parsed_messages:
    curr_time = datetime.strptime(msg['timestamp'], '%d/%m/%y %H:%M')
    curr_sender = msg['sender']

    if last_sender is not None and last_sender != curr_sender:
        gap_seconds = (curr_time - last_time).total_seconds()
        if gap_seconds > 0:
            response_gaps[curr_sender].append(gap_seconds)

    last_sender = curr_sender
    last_time = curr_time

avg_responses = {}
for name, gaps in response_gaps.items():
    avg_responses[name] = np.mean(gaps) if gaps else 0

silent_streaks = {name: 0 for name in unique_members}
start_date = d1.date()
end_date = d2.date()

for name in unique_members:
    max_streak = 0
    current_streak = 0
    curr_d = start_date

    active_days = set()
    for msg in parsed_messages:
        if msg['sender'] == name:
            msg_d = datetime.strptime(msg['timestamp'].split(' ')[0], '%d/%m/%y').date()
            active_days.add(msg_d)

    while curr_d <= end_date:
        if curr_d not in active_days:
            current_streak += 1
        else:
            if current_streak > max_streak:
                max_streak = current_streak
            current_streak = 0
        curr_d += timedelta(days=1)

    if current_streak > max_streak:
        max_streak = current_streak
    silent_streaks[name] = max_streak

print("Response patterns and historical streaks parsed successfully.")

Response patterns and historical streaks parsed successfully.


Feature 7 — Personality Archetype Detection

In [7]:
archetypes = {}

# 1. Calculate consecutive messaging bursts (Spammer logic)
consec_bursts = {name: [] for name in unique_members}
current_streak_count = 0
current_streak_user = None

for msg in parsed_messages:
    user = msg['sender']
    if user == current_streak_user:
        current_streak_count += 1
    else:
        if current_streak_user is not None:
            consec_bursts[current_streak_user].append(current_streak_count)
        current_streak_user = user
        current_streak_count = 1
if current_streak_user is not None:
    consec_bursts[current_streak_user].append(current_streak_count)

caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]

for name in unique_members:
    user_msgs = [m for m in parsed_messages if m['sender'] == name]
    total_user_msgs = len(user_msgs)

    if total_user_msgs == 0:
        archetypes[name] = "THE GHOST"
        continue

    # Night Owl check (23:00 to 04:59)
    night_msgs = 0
    for m in user_msgs:
        h = int(m['timestamp'].split(' ')[1].split(':')[0])
        if h >= 23 or h <= 4:
            night_msgs += 1
    night_ratio = night_msgs / total_user_msgs

    # Storyteller check (average message text words length)
    word_lengths = [len(m['text'].split()) for m in user_msgs if m['text'] != "<Media omitted>"]
    avg_words = np.mean(word_lengths) if word_lengths else 0

    # Drama Queen check (All uppercase strings or multiple exclamations)
    drama_count = 0
    for m in user_msgs:
        txt = m['text']
        if (txt.isupper() and len(txt) >= 3) or txt.count('!') >= 2:
            drama_count += 1
    drama_ratio = drama_count / total_user_msgs

    # Group Mom keyword frequency checks
    mom_score = 0
    for m in user_msgs:
        t_low = m['text'].lower()
        for kw in caring_keywords:
            mom_score += t_low.count(kw)

    avg_burst = np.mean(consec_bursts[name]) if consec_bursts[name] else 0

    # Strict fallback conditions matching the dataset thresholds
    if avg_burst > 3.5:
        archetypes[name] = f"THE SPAMMER (avg {avg_burst:.1f} msgs in a row)"
    elif night_ratio > 0.60:
        archetypes[name] = f"THE NIGHT OWL ({night_ratio * 100:.1f}% msgs after 11 PM)"
    elif avg_words > 30:
        archetypes[name] = f"THE STORYTELLER (avg {avg_words:.1f} words per msg)"
    elif drama_ratio > 0.30:
        archetypes[name] = f"THE DRAMA QUEEN ({drama_ratio * 100:.1f}% ALL-CAPS msgs)"
    elif silent_streaks[name] > 8:
        archetypes[name] = f"THE GHOST (silent {silent_streaks[name]} days straight)"
    elif mom_score > 10:
        archetypes[name] = f"THE GROUP MOM (caring score: {mom_score})"
    else:
        archetypes[name] = "THE COMEDIAN"

Feature 8 — The Final Report Formatting

In [12]:
# ==========================================
# FEATURE 8: THE FINAL REPORT FORMATTING
# ==========================================

print("=" * 60)
print(f'GROUPDNA REPORT — "{file_path.split("/")[-1].replace(".txt","")}"')
print(f"{total_days} days • {total_messages:,} messages • {len(unique_members)} members")
print("=" * 60)
print(f"Period       : {first_date_str} to {last_date_str}")
print(f"Busiest day  : {busiest_day[0]} ({busiest_day[1]} messages)")
print(f"Busiest hour : {busiest_hour[0]}:00 ({busiest_hour[1]} messages recorded overall)")
print("\nMESSAGES PER PERSON")

for name, count in sorted_participants:
    pct = (count / total_messages) * 100
    bar_len = int(pct / 2)
    bar_visual = "█" * bar_len
    print(f"{name:<8} {bar_visual:<15} {count} ({pct:.1f}%)")

print("\nACTIVITY HEATMAP (hour of day, columns 00 to 23)")
print("         00 02 04 06 08 10 12 14 16 18 20 22")
for name in unique_members:
    row_idx = member_to_idx[name]
    row_values = heatmap_matrix[row_idx]
    max_val = np.max(row_values) if np.max(row_values) > 0 else 1

    char_string = ""
    for val in row_values:
        ratio = val / max_val
        if ratio == 0: char_string += ". "
        elif ratio <= 0.25: char_string += ". "
        elif ratio <= 0.50: char_string += "░ "
        elif ratio <= 0.75: char_string += "▒ "
        else: char_string += "█ "
    print(f"{name:<8} {char_string}")

print("\nTHIS GROUP'S FAVORITE WORDS")
for word, count in sorted_words:
    word_bar = "█" * int(count / 20)
    print(f"{word:<8} {word_bar:<20} {count}")

print("\nRESPONSE PATTERNS")
fastest = min(avg_responses.items(), key=lambda x: x[1])
slowest = max(avg_responses.items(), key=lambda x: x[1])
print(f"Fastest replier : {fastest[0]} (avg {fastest[1]/60:.1f} minutes)")
print(f"Slowest replier : {slowest[0]} (avg {slowest[1]/3600:.1f} hours)")

print("\nLONGEST SILENT STREAKS")
for name in unique_members:
    print(f"{name:<8} : {silent_streaks[name]} days")

print("\nPERSONALITY ARCHETYPES")
for name in unique_members:
    print(f"{name:<8} → {archetypes[name]}")

print("=" * 60)
print("Generated by GroupDNA • Built with Python + NumPy")
print("=" * 60)

GROUPDNA REPORT — "hostel_bois"
60 days • 3,159 messages • 6 members
Period       : 01/04/24 to 30/05/24
Busiest day  : 04/05/24 (76 messages)
Busiest hour : 18:00 (246 messages recorded overall)

MESSAGES PER PERSON
Rahul    ██████████████  947 (30.0%)
Priya    ███████████     716 (22.7%)
Neha     ██████████      632 (20.0%)
Aman     ███████         488 (15.4%)
Karan    █████           352 (11.1%)
Vikas                    24 (0.8%)

ACTIVITY HEATMAP (hour of day, columns 00 to 23)
         00 02 04 06 08 10 12 14 16 18 20 22
Rahul    . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
Priya    . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
Neha     . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
Aman     ▒ █ █ ▒ █ . . . . . . . . . . . . . . . . . . ▒ 
Karan    . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
Vikas    . . . . . . . ░ █ ░ ░ . ▒ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒ 

THIS GROUP'S FAVORITE WORDS
how      ████████████████     321
guys     ███████████████      318
so       ██████████████